In [1]:
import datetime
import argparse
from pathlib import Path
import multiprocessing as mp
from functools import partial
import yaml
import h5py
import numpy as np
from tqdm import tqdm
from src.module.utils import parse_args, load_config, load_npy, multiprocess_function
from src.module.prepare_dataset import preprocess, prepare
from dataclasses import dataclass

In [2]:
from dataclasses import dataclass


@dataclass
class Arguments:
    conf: str = "../../config/conf.yaml"
    strategy: str = "hypophetversion2"
    min = None
    max = None

In [3]:
def set_environment(config):
    src_path = Path(config["data_path"]).expanduser() / "02_04.processed"
    save_path = Path(config["data_path"]).expanduser() / "02_04.processed"
    file_path = sorted(src_path.rglob(f"*{config['fs']}fs*processed.h5"))
    env = {"src_path": src_path, "save_path": save_path, "file_path": file_path}
    return env


def load_h5(file_path) -> tuple:
    with h5py.File(file_path, "r") as f:
        ts = f["signal/ts"][:]
        sig = f["signal/abp"][:]
        mbp = f["signal/mbp"][:]
        sqi = f["signal/sqi"][:]
        fs = f["signal"].attrs["fs"]
        sig_len = f["signal"].attrs["len"]
        cid = f.attrs["caseid"]

    return cid, fs, sig_len, ts, sig, mbp, sqi

In [4]:
args = Arguments()
config = load_config(args.conf)
env = set_environment(config)

In [76]:
file_path = env["file_path"][1]

In [77]:
cid, _, _, ts, sig, _, _ = load_h5(file_path)
pos_segments_idx = np.load(
    f"{str(file_path)[:-3]}_{args.strategy}_adj{config['adjacency']}_pred_pos_segments.npy"
)
pos_segments_idx = np.vstack([np.arange(*s) for s in pos_segments_idx])
neg_segments_idx = np.load(
    f"{str(file_path)[:-3]}_{args.strategy}_adj{config['adjacency']}_neg_segments.npy"
)
neg_segments_idx = np.vstack([np.arange(*s) for s in neg_segments_idx])
print(pos_segments_idx.shape, neg_segments_idx.shape)

(150, 3000) (58, 3000)


In [78]:
pos_segments = np.expand_dims(sig[pos_segments_idx], axis=-1)
neg_segments = np.expand_dims(sig[neg_segments_idx], axis=-1)
print(pos_segments.shape, neg_segments.shape)

(150, 3000, 1) (58, 3000, 1)


In [79]:
pos_ts = np.expand_dims(ts[pos_segments_idx], axis=-1)
neg_ts = np.expand_dims(ts[neg_segments_idx], axis=-1)
print(pos_ts.shape, neg_ts.shape)

(150, 3000, 1) (58, 3000, 1)


In [80]:
pos_cid = np.full(pos_segments.shape, [cid], dtype="<U4")
neg_cid = np.full(neg_segments.shape, [cid], dtype="<U4")
print(pos_cid.shape, neg_cid.shape)

(150, 3000, 1) (58, 3000, 1)


In [81]:
pos_data = np.concatenate([pos_cid, pos_ts, pos_segments], axis=-1, dtype=str)
neg_data = np.concatenate([neg_cid, neg_ts, neg_segments], axis=-1, dtype=str)
print(pos_data.shape, neg_data.shape)

(150, 3000, 3) (58, 3000, 3)


In [88]:
print(
    np.load("../../data/05.dataset/hypophetversion2_adj20_pos_dataset.npy").shape,
    np.load("../../data/05.dataset/hypophetversion2_adj20_neg_dataset.npy").shape,
)

(240, 3000, 3) (78, 3000, 3)
